# GPU pilot: Qwen3.5 on the 300-question pilot

Runs on **Google Colab** (Runtime → Change runtime type → **L4** or **A100**; T4 also works but is untested for Qwen3.5) or **Kaggle** (Accelerator → **GPU T4 x2**, *not* P100; Settings → Internet **on**).

What it does: starts a vLLM server with Qwen3.5 on this GPU, installs the `qdu` package with uv, downloads the pilot datasets, and runs the text / vision / hybrid pipelines on `benchmarks/pilot_300.csv` against `localhost` (so measured latency has no network time). Runs are resumable: if the session dies, run the cells again and finished calls are skipped.

Download `runs/<name>.zip` at the end and commit nothing from it by hand; the report is regenerated with `qdu benchmark report`.

In [ ]:
# 1. Settings
MODEL = "Qwen/Qwen3.5-4B"
RUN_NAME = "pilot-qwen3.5-4b"
REPO = "https://github.com/emre-geris/query-based-document-understanding.git"
# Private repo? Put a GitHub token in Colab Secrets / Kaggle Secrets as GITHUB_TOKEN.

import os, subprocess
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
os.chdir(WORK)
cap = subprocess.run(["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print(cap)
major = float(cap.splitlines()[0].split(",")[1])
DTYPE = "bfloat16" if major >= 8 else "float16"   # T4 (7.5) has no bf16
print("workdir:", WORK, "| dtype:", DTYPE)

In [ ]:
# 2. System packages: Tesseract (OCR for text/hybrid), uv, vLLM (takes a few minutes)
!apt-get -qq update && apt-get -qq install -y tesseract-ocr > /dev/null
!pip -q install -U uv vllm
!tesseract --version | head -1 && uv --version && python -c "import vllm; print('vllm', vllm.__version__)"

In [ ]:
# 3. Start the vLLM server in the background (first start downloads the model, ~9 GB)
import shlex, time, urllib.request
cmd = (f"nohup vllm serve {MODEL} --port 8000 --dtype {DTYPE} --max-model-len 16384 "
       f"--gpu-memory-utilization 0.90 --limit-mm-per-prompt '{{\"image\": 1}}' "
       f"> {WORK}/vllm.log 2>&1 &")
print(cmd)
subprocess.run(cmd, shell=True)
for i in range(120):
    try:
        urllib.request.urlopen("http://localhost:8000/v1/models", timeout=5)
        print(f"server ready after {i * 10} s"); break
    except Exception:
        time.sleep(10)
else:
    print("server did not start; last log lines:")
!tail -n 25 {WORK}/vllm.log

In [ ]:
# 4. Get the code and install it with uv (uv brings its own Python 3.13; vLLM stays in the system Python)
token = os.environ.get("GITHUB_TOKEN")
try:
    from google.colab import userdata; token = token or userdata.get("GITHUB_TOKEN")
except Exception:
    pass
try:
    from kaggle_secrets import UserSecretsClient; token = token or UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    pass
url = REPO.replace("https://", f"https://x-access-token:{token}@") if token else REPO
if not os.path.isdir("qdu-repo"):
    subprocess.run(["git", "clone", "-q", url, "qdu-repo"], check=True)
os.chdir(f"{WORK}/qdu-repo")
!git pull -q && git log --oneline -1
!uv sync -q && uv run qdu --help | head -3

In [ ]:
# 5. Datasets for the pilot (DocVQA, InfographicVQA, ChartQA -> HF cache, ~1.5 GB)
!uv run qdu data download --datasets docvqa,infographicvqa,chartqa

In [ ]:
# 6. Smoke test: 3 questions x 3 paths. Check that answers look like short spans, not reasoning.
SPEC = f"openai:{MODEL}"
!uv run qdu benchmark run --text-model {SPEC} --vision-model {SPEC} --limit 3 --name smoke-{RUN_NAME}
!uv run python -c "import json; [print(r['pipeline'], '|', r['prediction'], '| gold', r['answers'], '| tok', r['input_tokens'], r['output_tokens'], '| s', round(r['model_latency_s'] or 0, 2), r['error'] or '') for r in map(json.loads, open('runs/smoke-{RUN_NAME}/predictions.jsonl'))]"

In [ ]:
# 7. The pilot: 300 questions x 3 paths. Resumable: re-run this cell after a disconnect.
!uv run qdu benchmark run --text-model {SPEC} --vision-model {SPEC} --name {RUN_NAME}

In [ ]:
# 8. Report and package the run for download
!uv run qdu benchmark report runs/{RUN_NAME}
!cd runs && zip -qr {WORK}/{RUN_NAME}.zip {RUN_NAME} && ls -lh {WORK}/{RUN_NAME}.zip